# Try TariffTrail: Strands Decider 2B walking a tree, live

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahanteshimath/AWS-Strands-Decider-Hackathon/blob/main/notebooks/try_tarifftrail.ipynb)

This notebook runs the real **Strands Decider 2B** on a free Colab GPU and uses it the way
[TariffTrail](https://github.com/mahanteshimath/AWS-Strands-Decider-Hackathon) does: one calibrated choice per fork
of a tree, best-first on path probability, with `none_of_these` at every fork. **No AWS account is needed.**

1. **The pattern on a small tree:** route a request to 1 of 48 tools.
2. **A real 2026 customs ruling:** re-run the exact seeded search from the evaluation (the retrieval seeds are
   precomputed in the repo) and compare with the recorded result and CBP's answer.
3. **Your own product:** walk the 18,946-code US tariff schedule.

**Before you start:** *Runtime > Change runtime type > T4 GPU*. Setup downloads the 2B model and takes a few
minutes. After that, each Decider call takes well under a second on the GPU.

*Decision support, not legal advice: a licensed customs broker is responsible for every customs entry.*

## 1. Setup

The TariffTrail code, then Strands Decider in its own environment (the same steps as the GPU host in `infra/userdata.sh`).

In [ ]:
!git clone -q https://github.com/mahanteshimath/AWS-Strands-Decider-Hackathon /content/tarifftrail
%cd /content/tarifftrail
!pip -q install -e .
!python data/fetch_hts.py          # the 2026 US tariff schedule from USITC (public domain)

In [ ]:
# Strands Decider, installed into its own virtual environment so it does not disturb Colab's packages
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
!git clone -q https://github.com/strands-labs/strands-decider /content/strands-decider
!cd /content/strands-decider && ~/.local/bin/uv venv -q --python 3.12 /content/decider-venv && VIRTUAL_ENV=/content/decider-venv ~/.local/bin/uv pip install -q -e ".[cuda]"

In [ ]:
# Download the published Decider checkpoint. Colab's T4 has no native bfloat16, so load it in float32 there
# (2B parameters fit easily in 16 GB); GPUs from Ampere on keep the checkpoint's bfloat16.
import json, torch
from huggingface_hub import snapshot_download

ckpt = snapshot_download("StrandsAgents/strands-decider-2B-qwen3.5-v1-2610", local_dir="/content/decider-ckpt")
cfg_path = f"{ckpt}/strands_decider_config.json"
cfg = json.load(open(cfg_path))
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8:
    cfg["torch_dtype"] = "float32"
    json.dump(cfg, open(cfg_path, "w"), indent=2)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)",
      "| dtype:", cfg["torch_dtype"])

In [ ]:
# Serve Decider in the background on port 8099, where tariff_trail.decider looks by default.
import subprocess, time, urllib.request

device = "cuda" if torch.cuda.is_available() else "cpu"
log = open("/content/decider.log", "w")
server = subprocess.Popen(["/content/decider-venv/bin/strands-decider", "serve", ckpt, "--device", device,
                           "--port", "8099", "--model-name", "strands-decider-2B-qwen3.5-v1-2610"],
                          stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):  # the first start also downloads the Qwen3.5-2B base model
    try:
        urllib.request.urlopen("http://127.0.0.1:8099/health", timeout=2)
        print("Decider is up")
        break
    except Exception:
        if server.poll() is not None:
            break
        time.sleep(5)
else:
    print("still starting; re-run this cell")
if server.poll() is not None:
    print(open("/content/decider.log").read()[-3000:])

## 2. The pattern on a small tree: 1 of 48 tools

`examples/tool_catalog/catalog.json` has 8 services and 48 tools. Decider picks the service, then the tool.
The path probability decides whether an agent may call the tool directly (here: at least 0.6) or should ask first.

In [ ]:
import sys, json, time
sys.path.insert(0, "/content/tarifftrail")
sys.path.insert(0, "/content/tarifftrail/examples/tool_catalog")
from tariff_trail.decider import Decider
from tariff_trail.tree_descent import CalibratedTreeDescent, DictTree, Policy
from route import WORDING, state

catalog = DictTree(json.load(open("/content/tarifftrail/examples/tool_catalog/catalog.json")))
router = CalibratedTreeDescent(catalog, Decider(), policy=Policy(auto=0.6), wording=WORDING, render_state=state)

for request in ["Give the auditor a download link to report.pdf that expires in 24 hours",
                "Push tomorrow's design review to Friday 3pm",
                "Ping Priya privately that the build is green"]:
    started = time.perf_counter()
    r = router.descend(request)
    print(f"{request}\n  -> {r.leaf}   p = {r.p:.2f}   {'call it' if r.auto else 'ask the user first'}"
          f"   ({r.expansions} Decider calls, {time.perf_counter() - started:.1f} s)")
    for s in r.steps:
        top = sorted(s.options.items(), key=lambda kv: kv[1], reverse=True)[:3]
        print("     ", " | ".join(f"{k.rsplit('/', 1)[-1]} {v:.2f}" for k, v in top))

## 3. A real 2026 customs ruling, re-run live

Pick any of the 392 evaluation rulings (60 dev, 332 test). The search below is exactly the one from the
evaluation: the 8 candidate headings proposed by the 20 most similar pre-2026 rulings (precomputed with Titan
embeddings into `data/eval/seeds.jsonl`), then Decider's best-first walk. It prints every fork as Decider answers.

The full system then makes one Amazon Bedrock call to choose among Decider's top 3 codes plus the nearest ruling's
code; that step needs AWS, so here you see Decider's own answer next to what the evaluation recorded.

In [ ]:
import random
from tariff_trail.classify import TARIFF_POLICY
from tariff_trail.hts.questions import HTS_WORDING, hts_state
from tariff_trail.hts.tree import load

tree = load()
root = "/content/tarifftrail"
cases = {json.loads(l)["id"]: json.loads(l) for s in ("dev", "test") for l in open(f"{root}/data/eval/{s}.jsonl")}
seeds = {json.loads(l)["id"]: json.loads(l) for l in open(f"{root}/data/eval/seeds.jsonl")}
recorded = {json.loads(l)["id"]: json.loads(l) for l in open(f"{root}/eval/results/test/tariff_trail.jsonl")}


def label(node):
    if node == "seeded":
        return "headings proposed by similar past rulings"
    return tree.option_text(node).split(" (anything", 1)[0][:70] if node != tree.root_id else "the whole schedule"


def show_fork(event):
    top = sorted(event["probs"].items(), key=lambda kv: kv[1], reverse=True)[:3]
    names = [("none of these" if k == "none_of_these" else label(k)[:38]) for k, _ in top]
    print(f"  {label(event['node'])[:46]:46} | " + " | ".join(f"{n} {p:.2f}" for n, (_, p) in zip(names, top)))


def dotted(code):
    return ".".join(p for p in (code[:4], code[4:6], code[6:8], code[8:10]) if p)


def replay(case_id):
    case, seed = cases[case_id], seeds[case_id]
    print(f"Ruling {case_id}\n{case['facts']}\n")
    walker = CalibratedTreeDescent(tree, Decider(), policy=TARIFF_POLICY, wording=HTS_WORDING, render_state=hts_state,
                                   seed=lambda tree, facts: seed["headings"], observe=show_fork)
    started = time.perf_counter()
    result = walker.descend(case["facts"])
    print(f"\n{result.expansions} Decider calls in {time.perf_counter() - started:.1f} s")
    print("Decider's top codes:", ", ".join(f"{dotted(tree.code_of(c))} (p {p:.2f})" for c, p in result.candidates[:3]))
    print("Nearest past ruling:", dotted(seed["vote"]))
    if case_id in recorded:
        print("Recorded TariffTrail answer (after the Bedrock call):", dotted(recorded[case_id]["code"]))
    print("CBP's ruling:       ", dotted(case["gold"]))


replay("N357194")   # an adult Anubis costume; try random.choice(list(cases)) for another

## 4. Your own product

**With AWS credentials** (add `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` and, if you use temporary credentials,
`AWS_SESSION_TOKEN` under the key icon, *Secrets*, on the left; the account needs Amazon Bedrock access to Titan Text
Embeddings V2 and Amazon Nova Pro in us-east-1), this cell runs the **full TariffTrail system**: it downloads the
prebuilt Titan index of 7,024 past rulings from the repo's release, seeds the walk with their headings, and makes one
Bedrock call to choose among real codes. Expect about one product in three exactly right at 10 digits (the evaluated
rate); that is why every answer goes to a broker, with its alternatives.

**Without credentials**, it seeds the walk with an offline keyword match over heading wording instead. That is much
weaker than retrieval (the right heading is among its 8 proposals for 42% of test products, against 78% for
retrieval), so treat the answer as a demonstration of the search, not a classification.

In [ ]:
import os, subprocess
os.environ.setdefault("AWS_REGION", "us-east-1")
os.environ.setdefault("TT_REASONER_MODEL", "us.amazon.nova-pro-v1:0")  # the model the evaluation used
try:
    from google.colab import userdata
    for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass
import boto3
have_aws = boto3.Session().get_credentials() is not None

product = "Men's five-pocket jeans, woven denim, 98% cotton 2% elastane"  # change me

if have_aws:
    subprocess.run([sys.executable, "data/fetch_precedents.py"], check=True)
    from tariff_trail.classify import TariffTrail
    c = TariffTrail(tree=tree, decider=Decider()).classify(product, observe=show_fork)
    print(f"\n{c.htsno}   general duty {c.duty or '-'}   "
          f"{'FAST-TRACK review (Decider, Bedrock and past rulings agree)' if c.fast_track else 'BROKER review (the sources disagree)'}")
    print("Alternatives:", ", ".join(x["htsno"] for x in c.candidates if x["code"] != c.code)[:120] or "none")
    if c.adjudications:
        print("Bedrock's reason:", c.adjudications[-1]["reason"])
    print(c.path_text)
else:
    from tariff_trail.simulate import KeywordScorer, tokens

    scorer = KeywordScorer(tree)
    headings = [n for n in tree.nodes if n.startswith("H:")]

    def keyword_seed(tree, facts, n=8):
        words = set(tokens(facts))
        return sorted(headings, key=lambda h: scorer.score(h, words), reverse=True)[:n]

    print("No AWS credentials: keyword-seeded Decider walk (a demonstration, not a classification)\n")
    walker = CalibratedTreeDescent(tree, Decider(), policy=TARIFF_POLICY, wording=HTS_WORDING, render_state=hts_state,
                                   seed=keyword_seed, observe=show_fork)
    result = walker.descend(product)
    print("\nTop codes:", ", ".join(f"{dotted(tree.code_of(c))} (p {p:.2f})" for c, p in result.candidates[:3]))
    print(tree.path_text(result.leaf))

## Where to go next

- **The pattern in any Strands agent:** `tariff_trail/strands_kit.py` gives you a decision tool and a gate
  (`DecisionGate`, a Strands `InterventionHandler`) that denies made-up items and pauses for a person below the bar.
  See `examples/tool_catalog/agent.py` (needs Amazon Bedrock for the agent's model).
- **The full TariffTrail system** (Titan retrieval over 7,024 past rulings + one Bedrock call + the filing gate):
  README, "Classify live".
- **How it was evaluated:** `eval/RESULTS.md` and `eval/ABLATION.md`.

In [ ]:
server.terminate()  # stop the Decider server when you are done